# ♻️ CO₂ Emissions & Recycling Impact Analysis
**Author:** Mayowa Alamutu  
**Domain:** Environment / Sustainability / Circular Economy  
**Tools:** Python, Pandas, Seaborn, Matplotlib, Excel (impact calculator)  
**Research Link:** Akinlabi et al. (2025) — *"Recycling and Reuse of Materials in Metallurgical Processes"*, Harvard IJERT  
DOI: https://doi.org/10.70382/hijert.v8i5.004  

---

## Project Overview
Directly inspired by my co-authored publication on metallurgical recycling, this notebook quantifies  
the **environmental benefit of recycling** across five key metal streams in Nigeria:  
Aluminium, Steel, Copper, Lead (batteries), and E-waste metals.

**Key Question:** How much CO₂ and energy are we saving — or losing — by not formalising Nigeria's recycling sector?


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'monospace'
print("✅ Libraries ready")

## 2. Emission Factor Database
> *Sourced from IEA (2023), World Steel Association, International Aluminium Institute, and IPCC AR6.*

In [ ]:
# CO2 emission factors (kg CO2 per tonne of metal produced)
emission_factors = pd.DataFrame({
    'metal':            ['Aluminium', 'Steel', 'Copper', 'Lead', 'E-waste metals'],
    'primary_co2':      [11500, 1850, 4100, 1700, 6000],   # virgin production
    'recycled_co2':     [700,    776,  615,   340,   900],  # secondary/recycled
    'energy_saving_pct':[95,     60,   85,    65,    78],   # % energy saved
    'nigeria_vol_tpa':  [18400, 142000, 6200, 22000, 4800],  # tonnes/yr recycled in Nigeria
    'color':            ['#FF6B35','#1E88E5','#FF9800','#8D6E63','#66BB6A']
})

emission_factors['co2_saved_per_tonne'] = (
    emission_factors['primary_co2'] - emission_factors['recycled_co2']
)
emission_factors['co2_reduction_pct'] = (
    emission_factors['co2_saved_per_tonne'] / emission_factors['primary_co2'] * 100
).round(1)
emission_factors['total_co2_saved_tonnes'] = (
    emission_factors['co2_saved_per_tonne'] * emission_factors['nigeria_vol_tpa'] / 1000
).round(0)

print("Emission Factors & Impact Summary:")
print(emission_factors[['metal','primary_co2','recycled_co2',
                          'co2_reduction_pct','total_co2_saved_tonnes']].to_string(index=False))
print(f"\nTotal CO₂ saved (current volumes): {emission_factors['total_co2_saved_tonnes'].sum():,.0f} tonnes/yr")

## 3. Visualise — CO₂ & Energy Savings

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Environmental Impact of Metal Recycling in Nigeria', fontsize=14, fontweight='bold')

metals = emission_factors['metal']
x = np.arange(len(metals))
width = 0.35

# ── Primary vs Recycled CO2 ──────────────────────────────────────────
axes[0].bar(x - width/2, emission_factors['primary_co2'], width,
             label='Primary Production', color='#EF5350', alpha=0.85)
axes[0].bar(x + width/2, emission_factors['recycled_co2'], width,
             label='Recycled',           color='#66BB6A', alpha=0.85)
axes[0].set_title('CO₂: Primary vs. Recycled\n(kg CO₂ per tonne)', fontsize=11)
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Al','Steel','Cu','Pb','E-waste'], fontsize=10)
axes[0].set_ylabel('kg CO₂ / tonne')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'{v/1000:.0f}k'))

# ── Energy saving % ──────────────────────────────────────────────────
ef_sorted = emission_factors.sort_values('energy_saving_pct', ascending=True)
bars = axes[1].barh(ef_sorted['metal'], ef_sorted['energy_saving_pct'],
                     color=ef_sorted['color'])
axes[1].set_title('Energy Saving vs. Primary Production (%)', fontsize=11)
axes[1].set_xlabel('Energy Saved (%)')
axes[1].set_xlim(0, 105)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, ef_sorted['energy_saving_pct']):
    axes[1].text(val + 1, bar.get_y() + bar.get_height()/2,
                  f'{val}%', va='center', fontsize=10, fontweight='bold')

# ── Total CO2 saved per metal ─────────────────────────────────────────
axes[2].bar(metals, emission_factors['total_co2_saved_tonnes'],
             color=emission_factors['color'])
axes[2].set_title('Total CO₂ Saved — Nigeria\n(Current Recycling Volumes)', fontsize=11)
axes[2].set_ylabel('CO₂ Saved (tonnes/yr)')
axes[2].set_xticklabels(['Al','Steel','Cu','Pb','E-waste'], fontsize=10)
axes[2].yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'{v/1000:.0f}k'))
for i, val in enumerate(emission_factors['total_co2_saved_tonnes']):
    axes[2].text(i, val + 1500, f'{val/1000:.0f}k t', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('co2_impact.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Scenario Analysis — What If We Recycled More?

In [ ]:
# Current avg recycling rate in Nigeria's formal sector
current_rate_pct = 13.4   # % (from our trend data)
target_rates = np.arange(10, 91, 5)

# Estimated total metal waste generated in Lagos per year (tonnes)
total_waste = {
    'Aluminium':      180000,
    'Steel':          850000,
    'Copper':          42000,
    'Lead':           140000,
    'E-waste metals':  38000
}

scenarios = []
for rate in target_rates:
    total_co2_saved = 0
    total_energy_gwh = 0
    for _, row in emission_factors.iterrows():
        vol = total_waste[row['metal']] * rate / 100
        co2  = vol * row['co2_saved_per_tonne'] / 1000          # tonnes CO2
        # energy saving estimate: primary energy ~15 MWh/t Al, ~5 MWh/t steel
        energy_factors = {'Aluminium':15,'Steel':5,'Copper':12,'Lead':4,'E-waste metals':8}
        energy = vol * energy_factors[row['metal']] * row['energy_saving_pct']/100 / 1000  # GWh
        total_co2_saved  += co2
        total_energy_gwh += energy
    scenarios.append({
        'recycling_rate': rate,
        'co2_saved_kt': total_co2_saved / 1000,
        'energy_saved_gwh': total_energy_gwh
    })

scenario_df = pd.DataFrame(scenarios)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Scenario Analysis: Impact of Increasing Recycling Rate', fontsize=13, fontweight='bold')

# CO2 savings
axes[0].fill_between(scenario_df['recycling_rate'],
                      scenario_df['co2_saved_kt'], alpha=0.2, color='green')
axes[0].plot(scenario_df['recycling_rate'], scenario_df['co2_saved_kt'],
              'g-o', lw=2.5, ms=5)
axes[0].axvline(current_rate_pct, color='orange', linestyle='--', lw=1.5,
                 label=f'Current rate ({current_rate_pct}%)')
axes[0].axvline(86, color='blue', linestyle='--', lw=1.5, label='EU avg (86%)')
axes[0].set_title('CO₂ Saved vs. Recycling Rate')
axes[0].set_xlabel('Formal Recycling Rate (%)')
axes[0].set_ylabel('CO₂ Saved (000 tonnes/yr)')
axes[0].legend(fontsize=9)

# Energy savings
axes[1].fill_between(scenario_df['recycling_rate'],
                      scenario_df['energy_saved_gwh'], alpha=0.2, color='teal')
axes[1].plot(scenario_df['recycling_rate'], scenario_df['energy_saved_gwh'],
              color='teal', marker='o', lw=2.5, ms=5)
axes[1].axvline(current_rate_pct, color='orange', linestyle='--', lw=1.5,
                 label=f'Current ({current_rate_pct}%)')
axes[1].set_title('Energy Recovered vs. Recycling Rate')
axes[1].set_xlabel('Formal Recycling Rate (%)')
axes[1].set_ylabel('Energy Recovered (GWh/yr)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('scenario_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Digitalization vs. Recovery Efficiency (From Research Data)

In [ ]:
# Data from our paper — 10 Lagos recycling facilities
facility_data = pd.DataFrame({
    'facility_id':         ['F1','F2','F3','F4','F5','F6','F7','F8','F9','F10'],
    'digitalization_score':[5,   4,   3,   4,   5,   0,   0,   0,   0,   0  ],
    'recovery_efficiency': [91.2,88.6,86.3,89.5,92.0,76.4,74.8,77.1,75.5,73.9],
    'energy_kwh_per_tonne':[840, 880, 910, 870, 825, 1120,1155,1105,1130,1175],
    'co2_kg_per_tonne':    [220, 245, 260, 230, 210, 330, 345, 320, 335, 360],
    'type': ['Digitized']*5 + ['Non-digitized']*5
})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Digitalization Impact on Recycling Performance', fontsize=14, fontweight='bold')
colors = facility_data['type'].map({'Digitized':'#43A047','Non-digitized':'#EF5350'})

# Scatter — digitalization vs recovery
axes[0].scatter(facility_data['digitalization_score'],
                 facility_data['recovery_efficiency'],
                 c=colors, s=120, edgecolors='white', lw=1.5, alpha=0.9)
# Trend line
z = np.polyfit(facility_data['digitalization_score'], facility_data['recovery_efficiency'], 1)
xfit = np.linspace(0, 5, 50)
axes[0].plot(xfit, np.polyval(z, xfit), 'k--', lw=1.5, alpha=0.6)
corr_r = facility_data['digitalization_score'].corr(facility_data['recovery_efficiency'])
axes[0].set_title(f'Digitalization → Recovery  (r = {corr_r:.2f})')
axes[0].set_xlabel('Digitalization Score (0–5)')
axes[0].set_ylabel('Metal Recovery Efficiency (%)')

# Energy consumption by type
axes[1].boxplot(
    [facility_data[facility_data['type']=='Digitized']['energy_kwh_per_tonne'],
     facility_data[facility_data['type']=='Non-digitized']['energy_kwh_per_tonne']],
    labels=['Digitized','Non-digitized'],
    patch_artist=True,
    boxprops=dict(facecolor='#43A047', alpha=0.5)
)
axes[1].set_title('Energy Consumption by Digitalization')
axes[1].set_ylabel('Energy (kWh per tonne)')

# CO2 by type
axes[2].boxplot(
    [facility_data[facility_data['type']=='Digitized']['co2_kg_per_tonne'],
     facility_data[facility_data['type']=='Non-digitized']['co2_kg_per_tonne']],
    labels=['Digitized','Non-digitized'],
    patch_artist=True,
    boxprops=dict(facecolor='#EF5350', alpha=0.4)
)
axes[2].set_title('CO₂ Emissions by Digitalization')
axes[2].set_ylabel('CO₂ (kg per tonne)')

plt.tight_layout()
plt.savefig('digitalization_impact.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Final summary
print("=" * 62)
print("  CO₂ & RECYCLING IMPACT — KEY FINDINGS SUMMARY")
print("=" * 62)

total_now = emission_factors['total_co2_saved_tonnes'].sum()
full_formal = scenario_df[scenario_df['recycling_rate']==85]['co2_saved_kt'].values[0] * 1000
energy_full = scenario_df[scenario_df['recycling_rate']==85]['energy_saved_gwh'].values[0]

print(f"\n  CURRENT STATE (13.4% formal recycling rate):")
print(f"  • CO₂ saved annually        : {total_now:,.0f} tonnes")
print(f"  • Best performing metal     : Aluminium (-94% CO₂ vs primary)")
print(f"  • Worst CO₂ offender        : Non-recycled aluminium (11,500 kg/t)")

print(f"\n  IF RECYCLING RATE REACHED 85% (EU level):")
print(f"  • Additional CO₂ avoided    : ~{full_formal:,.0f} tonnes/yr")
print(f"  • Energy recovered          : ~{energy_full:.0f} GWh/yr")
print(f"  • Equivalent to powering    : ~{energy_full/0.3:.0f}k Nigerian homes")

print(f"\n  DIGITALIZATION FINDING (from research data):")
corr_d = facility_data['digitalization_score'].corr(facility_data['recovery_efficiency'])
corr_e = facility_data['digitalization_score'].corr(-facility_data['energy_kwh_per_tonne'])
print(f"  • Digitalization–Recovery correlation  : r = {corr_d:.2f} (strong positive)")
print(f"  • Digitalized facilities use {840:.0f} vs {1137:.0f} kWh/tonne")
print(f"  • = {(1137-840)/1137:.0%} less energy per tonne processed")
print("=" * 62)

## 6. Next Steps
- Connect to live NERC / Lagos State EPA data feeds  
- Build full Tableau dashboard (4 pages: overview, metal breakdown, zonal, 2030 projection)  
- Add Monte Carlo simulation for recycling scenario uncertainty  
- Publish updated findings as a data brief  

---
*Notebook by Mayowa Alamutu | [Portfolio](https://mayowahabeeb.framer.website/) | [LinkedIn](https://www.linkedin.com/in/mayowa-alamutu-84185a25a)*  
*Research: Akinlabi et al. (2025). Harvard IJERT. DOI: 10.70382/hijert.v8i5.004*
